# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Inspect available record sets and their fields using @id
print("\nAvailable Record Sets:")
record_sets = dataset.metadata.record_sets
for idx, rset in enumerate(record_sets):
    print(f"  [{idx}] Name: {rset['name']} | @id: {rset['@id']}")

print("\nFields per Record Set:")
for rset in record_sets:
    print(f"Record Set: {rset['name']} (@id: {rset['@id']})")
    for field in rset['fields']:
        print(f"    Field: {field['name']} (@id: {field['@id']}) | type: {field.get('dataType', 'Unknown')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose all record set @id values (shown in previous cell)
record_sets_ids = [rset['@id'] for rset in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from record set {record_set_id}")

# For demonstration, focus exploration on the first record set
main_rs_id = record_sets_ids[0]
print(f"Fields in the main record set: {dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# List numeric fields in the main record set for EDA
df = dataframes[main_rs_id]
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric fields: {numeric_fields}")

# Example: Choose the first numeric field for demonstration (e.g., coverage_percent, MW, peptide_count)
if len(numeric_fields) > 0:
    numeric_field = numeric_fields[0]
else:
    raise Exception("No numeric fields available for EDA.")

threshold = df[numeric_field].mean()
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a group field if any exists (e.g., by 'sample' or a string field)
string_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
if string_fields:
    group_field = string_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example visualization with matplotlib and seaborn
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field)
plt.title(f'Distribution of {numeric_field}')
plt.show()

# If grouped data is available, show a plot by group
if string_fields:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f'{numeric_field} by {group_field}')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use `mlcroissant` to load, inspect, and analyze the FAIR² mass spectrometry dataset. We explored data attributes, filtered records, normalized numeric values, grouped by key string fields, and produced basic visualizations. 

Further exploration can be done by leveraging additional record sets or examining correlations between additional numeric and categorical fields. For more advanced analytics, consider leveraging domain-specific knowledge regarding protein mass spectrometry and associated biological context.